In [2]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/09 18:11:06 WARN Utils: Your hostname, pc-gaymer, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/03/09 18:11:06 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 18:11:11 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Question 1:

In [3]:
spark.version

'4.1.1'

Question 2:

In [6]:
df_yellow = spark.read.parquet('yellow_tripdata_2025-11.parquet')
df_yellow.repartition(4).write.parquet('yellow_tripdata_2025-11_partitioned', mode='overwrite')
import os

folder = 'yellow_tripdata_2025-11_partitioned'
files = [f for f in os.listdir(folder) if f.endswith('.parquet')]
sizes = [os.path.getsize(os.path.join(folder, f)) for f in files]

for f, s in zip(files, sizes):
    print(f"{f}: {s / 1024 / 1024:.2f} MB")

print(f"\nAverage size: {sum(sizes) / len(sizes) / 1024 / 1024:.2f} MB")

part-00000-b372dd0d-c2a8-4f80-bd18-4f8cac9b51db-c000.snappy.parquet: 24.41 MB
part-00001-b372dd0d-c2a8-4f80-bd18-4f8cac9b51db-c000.snappy.parquet: 24.41 MB
part-00002-b372dd0d-c2a8-4f80-bd18-4f8cac9b51db-c000.snappy.parquet: 24.42 MB
part-00003-b372dd0d-c2a8-4f80-bd18-4f8cac9b51db-c000.snappy.parquet: 24.42 MB

Average size: 24.42 MB


Question 3:

In [7]:
from pyspark.sql import functions as F

df_yellow.filter(F.to_date(F.col('tpep_pickup_datetime')) == '2025-11-15').count()


162604

Question 4:

In [8]:
from pyspark.sql import functions as F

df_yellow.withColumn(
    'duration_hours',
    (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600
).agg(F.max('duration_hours')).collect()[0][0]


90.64666666666666

Question 5:

http://localhost:4040

Question 6:

In [9]:
zones = spark.read.option("header", True).option("inferSchema", True).csv("zones/taxi_zone_lookup.csv")

least_frequent_zone = (
    df_yellow.groupBy("PULocationID")
    .count()
    .join(zones, df_yellow.PULocationID == zones.LocationID, "left")
    .orderBy("count")
    .select("Zone", "PULocationID", "count")
    .limit(1)
)

least_frequent_zone.show(truncate=False)

+---------------------------------------------+------------+-----+
|Zone                                         |PULocationID|count|
+---------------------------------------------+------------+-----+
|Governor's Island/Ellis Island/Liberty Island|105         |1    |
+---------------------------------------------+------------+-----+

